In [34]:
import os
import math
import numpy as np
from tqdm.auto import tqdm

from Bio.PDB import PDBParser
import py3Dmol

import torch

In [35]:
import utils_infer
from utils_infer import dict_load, dict_save, inference_pdb_write, pdb_line_write, RESIDUE_dict

from diffab.models import get_model

from diffab.modules.common.geometry import apply_rotation_to_vector, quaternion_1ijk_to_rotation_matrix, reconstruct_backbone, reconstruct_backbone_partially
from diffab.modules.common.so3 import so3vec_to_rotation, rotation_to_so3vec, random_uniform_so3

# Utility Functions

In [33]:
def binder_batch_prepare(
    structure_dict, epitope_dict = {}, binder_size = 200, print_shape = False,
):
    """
    Input:
        target data
    Output:
        processed binder data
        placeholder for the binder
    """
    batch = {}
    feat_list = [
        'aa',
        'pos_heavyatom',
        'mask_heavyatom',
        'fragment_type',  ## flagment token: 1 for antigen, 2 for target, 3 for scaffold, 4 for epitope
        'mask',
        'generate_flag',
        'chain_nb',
        'resi_nb',
    ]
    for feat in feat_list:
        batch[feat] = []
    
    ###################################
    # Target
    ###################################

    ###### chain-wise process ######
    chain_nb = 0
    
    for chain_id in structure_dict:

        if chain_id in epitope_dict:
            epitope_set = epitope_dict[chain_id]
        else:
            epitope_set = set()
        
        ### residue-wise process ###
        res_idx = 0 
        for r_i, res in enumerate(structure_dict[chain_id]):
            ### placeholder
            batch['aa'].append(res['aa'])
            coor = torch.zeros(4, 3)
            pos_mask = torch.zeros(4)
            ### coordinates and masks
            mask = 0
            for _i, atom in enumerate(['N', 'CA', 'C', 'O']):
                if atom in res['coor']:
                    coor[_i] = torch.from_numpy(res['coor'][atom])
                    pos_mask[_i] = 1
                    mask = 1
            batch['pos_heavyatom'].append(coor)
            batch['mask_heavyatom'].append(pos_mask)
            batch['mask'].append(mask)
            ### indexes
            batch['chain_nb'].append(chain_nb)
            batch['resi_nb'].append(res_idx)
            ### feature indicator
            batch['generate_flag'].append(0)
            # epitope
            if res['id'] in epitope_set:
                batch['fragment_type'].append(4)
            # other region of the traget
            else:
                batch['fragment_type'].append(1)
            res_idx += 1

        chain_nb += 1
    
    ###################################
    # Binder
    ###################################

    res_idx = 0
    for _ in range(binder_size):
    
        batch['aa'].append(0)
        batch['pos_heavyatom'].append(torch.zeros(4, 3))
        batch['mask_heavyatom'].append(torch.ones(4))
        batch['mask'].append(1)
        batch['chain_nb'].append(chain_nb)
        batch['resi_nb'].append(res_idx)
        ### 1 for target
        batch['generate_flag'].append(1)
        ### 2 for target, 3 for scaffold
        batch['fragment_type'].append(2)
        res_idx += 1
        
    ###################################
    # final out
    ###################################
    
    for feat in feat_list:  
        if 'heavyatom' in feat:
            batch[feat] = torch.stack(batch[feat])
        else:
            batch[feat] = torch.tensor(batch[feat])
        batch[feat] = batch[feat].unsqueeze(0)
        if print_shape:
            print(feat, batch[feat].shape)

    return batch

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [ ]:
def model_loading(model_path):

    checkpoint = torch.load(model_path, weights_only = False)
    print(model_path)
    print(checkpoint.keys())
    
    config = checkpoint['config']
    model = get_model(config.model)
    
    parameter_dict = {}
    for key in checkpoint['model'].keys():
        key_new = key
    
        if key.startswith('module'):
            key_new = key[7:]
    
        key_new = key_new.split('.')
        key_new_last = []
        for token in key_new:
            if token in {'spatial_coef', 'proj_query_point', 'proj_key_point'}:
                token = token + '_intra'
            key_new_last.append(token)
        key_new = '.'.join(key_new_last)
        parameter_dict[key_new] = checkpoint['model'][key]
    
    model.load_state_dict(parameter_dict)
    
    return model

In [44]:
def seq_recover(aa:torch.Tensor, length:int = None) -> str:
    """Recover sequence from the tensor.

    Args:
        aa: embedded sequence tensor; (L,).
        length: length of the sequence; if None consider the paddings.

    Return:
        seq: recovered sequence string. 
    """

    length = aa.shape[0] if length is None else min(length, aa.shape[0])
    seq = ''
    for i in range(length):
        idx = int(aa[i])
        if idx > 20:
            print('Error! Index %d is larger than 20.'%idx)
            break
        seq += ressymb_order[idx]
    return seq

def pdb_write(
    coor, path, seq, chain_nb, chain_list = 'ABCDEFG',
    atom_list = ['N', 'CA', 'C', 'O'],
    print_seq = False,
):
    seq_dict = {}
    with open(path, 'w') as wf:
        a_idx = 0

        for i, resi in enumerate(seq):
            ### residue-wise info
            r_idx = i + 1
            aa = RESIDUE_dict[resi]
            chain = chain_list[int(chain_nb[i])]
            if chain not in seq_dict:
                seq_dict[chain] = ''
            seq_dict[chain] += resi

            for j,vec in enumerate(coor[i]):
                ### atom-wise info
                atom = atom_list[j]
                a_idx += 1
                pdb_line_write(chain, aa, r_idx, atom, a_idx, vec, wf)

    if print_seq:
        for chain in seq_dict:
            print(f'*********** {chain} **************')
            print(seq_dict[chain])

In [49]:
def inference(
    model, batch, device = 'cuda', attempts = 10, 
    result_path = '../results/RBX1Binder/dict/debug.pkl',
    save_pdb = True,
    pdb_dir = '../results/RBX1Binder/pdbs/debug/',
    save_name = 'debug', 
    print_seq = False,
    chain_list = 'AB',
):
    os.makedirs(pdb_dir, exist_ok=True)
    model = model.to(device)

    ###### centralization ######
    mean = batch['pos_heavyatom'].sum(dim = (1, 2))   # (N, 3)
    mean = mean / batch['mask_heavyatom'].sum(dim = (1, 2)).unsqueeze(1)  # (N, 3)
    batch['pos_heavyatom'] -= mean.unsqueeze(1).unsqueeze(1)  # (N, L, 15, 3)
    batch['pos_heavyatom'][batch['mask_heavyatom'] == 0] = 0

    ###### device & masks ######
    feat_list = ['aa', 'pos_heavyatom', 'generate_flag', 'mask', 'mask_heavyatom', 'resi_nb', 'chain_nb', 'fragment_type']
    for key in feat_list:
        batch[key] = batch[key].to(device)
    batch['generate_flag'] = batch['generate_flag'].bool()
    batch['mask'] = batch['mask'].bool()
    batch['mask_heavyatom'] = batch['mask_heavyatom'].bool()

    out_dict = {}
    sample_idx = 1

    ### inference
    for attp in tqdm(range(attempts)):
        ### inference
        traj_batch = model.sample(batch = batch)

        ### feature transform
        t = 0
        R = so3vec_to_rotation(traj_batch[t][0])
        aa_new = traj_batch[t][2].cpu()   # t: sampling step. 2: Amino acid.
        bb_coor_batch, mask_atom_new = reconstruct_backbone_partially(
            pos_ctx = batch['pos_heavyatom'].cpu(),
            R_new = R.cpu(),
            t_new = traj_batch[t][1].cpu(),
            aa = aa_new,
            chain_nb = batch['chain_nb'].cpu(),
            res_nb = batch['resi_nb'].cpu(),
            mask_atoms = batch['mask_heavyatom'].cpu(),
            mask_recons = batch['generate_flag'].cpu(),
        )  # (N, L_max, 4, 3), _

        ### feature out
        length = batch['mask'].shape[1]
        for i, bb_coor in enumerate(bb_coor_batch):
            ### sample-wise process
            seq = seq_recover(aa_new[i], length = length)
            out_dict[sample_idx] = {
                'coor_true': batch['pos_heavyatom'][i][:length].cpu(),
                'aa_true': batch['aa'][i][:length].cpu(),
                'coor': bb_coor[:length],
                'seq': seq,
                'fragment_type': batch['fragment_type'][i][:length].cpu(),
                #'linker_mask': None if linker_mask is None else linker_mask[i][:length]
            }
            sample_idx += 1

    ###### mv feature and model to CPU ######
    for key in feat_list:
        batch[key] = batch[key].to(device)
    model = model.cpu()

    ### result save
    _ = dict_save(out_dict, result_path)
    if save_pdb:
        for attp in range(1, attempts + 1):
            pdb_write(
                coor = out_dict[attp]['coor'], 
                path = os.path.join(pdb_dir, '%s_attp%d.pdb' % (save_name, attp)), 
                seq = out_dict[attp]['seq'], 
                chain_nb = batch['chain_nb'][0],
                chain_list = chain_list,
                print_seq = print_seq,
            )
    
    return out_dict


# Data

In [2]:
parser = PDBParser(PERMISSIVE=True)

In [3]:
pdb_id = '2LGV'
path = '../data/RBX1_binder/%s.pdb' % pdb_id
structure_all = parser.get_structure(pdb_id, path)

### Load the basic information

In [5]:
ressymb_order = 'ACDEFGHIKLMNPQRSTVWYX'
RESIDUE_reverse_dict = {'ALA':'A', 'ARG':'R', 'ASN':'N', 'ASP':'D', 'CYS':'C', 'GLN':'Q', 'GLU':'E',
                        'GLY':'G', 'HIS':'H', 'ILE':'I', 'LEU':'L', 'LYS':'K', 'MET':'M', 'PHE':'F',
                        'PRO':'P', 'SER':'S', 'THR':'T', 'TRP':'W', 'TYR':'Y', 'VAL':'V', 'ASX':'B',
                        'GLX':'Z', 'UNK':'X'}
res_type_dict = {}
for i, char in enumerate(ressymb_order):
    res_type_dict[char] = i

In [24]:
structure_dict_all = {}

for model_id in range(3):
    
    structure = structure_all[model_id]
    structure_dict = {}
    
    for chain in structure:
        chain_id = chain.get_id()
        structure_dict[chain_id] = []
    
        for res in chain:
            res_id = res.get_id()
            if res_id[0] != ' ':
                continue
    
            res_idx = (str(res_id[1]) + res_id[2]).strip(' ')
            res_name = res.resname
            if res_name in RESIDUE_reverse_dict:
                res_letter = RESIDUE_reverse_dict[res_name]
            else:
                res_letter = 'X'
                
            res_info = {
                'id': res_idx, 
                'type': res_name, 
                'letter': res_letter,
                'aa': res_type_dict[res_letter],
                'coor': {}
            }
            for atom in ['N', 'CA', 'C', 'O', 'CB']:
                if atom in res.child_dict:
                    res_info['coor'][atom] = res.child_dict[atom].get_coord()
                elif atom != 'CB':
                    print(f'{atom} not found for {chain_id}-{res_idx}.')
                    
            structure_dict[chain_id].append(res_info)

    structure_dict_all[model_id] = structure_dict

In [25]:
for model_id in structure_dict_all:

    print(f'###################### Model {model_id} ######################')
    structure_dict = structure_dict_all[model_id]
    
    for chain in structure_dict:
        print(f'**************** {chain} *****************')
        id_list = [res['id'] for res in structure_dict[chain]]
        print(id_list)

    print()

###################### Model 0 ######################
**************** A *****************
['9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108']

###################### Model 1 ######################
**************** A *****************
['9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43'

### Loadable data

* aa               # (L,)  
* pos_heavyatom    # (L, 4, 3), backbone atom coors 
* mask_heavyatom   # (L, 4)
* fragment_type    # (L,)
* mask             # (L,)
* generate_flag    # (L,)
* chain_nb         # (L,)
* resi_nb          # (L,)

In [46]:
### eitope: A43–R46 / I54–E55–Q57 / W87–R91–L96
data_dict = {}

for epitope in [(43,46), (54, 57), (87, 96)]:
    epitope_dict = {'A': {str(i) for i in range(epitope[0], epitope[1] + 1)}}

    for model_id in structure_dict_all:
        structure_dict = structure_dict_all[model_id]

        for binder_size in range(100, 251, 50):
            data_name = f'{pdb_id}-{model_id}_epi{epitope[0]}-{epitope[1]}_L{binder_size}'
            print(data_name)
    
            data_dict[data_name] = binder_batch_prepare(
                structure_dict, 
                epitope_dict = epitope_dict, 
                binder_size = binder_size,
                print_shape = False,
            )

print()
print(f'{len(data_dict)} data versions.')
for feat in data_dict[data_name]:
    print(feat, data_dict[data_name][feat].shape)

2LGV-0_epi43-46_L100
2LGV-0_epi43-46_L150
2LGV-0_epi43-46_L200
2LGV-0_epi43-46_L250
2LGV-1_epi43-46_L100
2LGV-1_epi43-46_L150
2LGV-1_epi43-46_L200
2LGV-1_epi43-46_L250
2LGV-2_epi43-46_L100
2LGV-2_epi43-46_L150
2LGV-2_epi43-46_L200
2LGV-2_epi43-46_L250
2LGV-0_epi54-57_L100
2LGV-0_epi54-57_L150
2LGV-0_epi54-57_L200
2LGV-0_epi54-57_L250
2LGV-1_epi54-57_L100
2LGV-1_epi54-57_L150
2LGV-1_epi54-57_L200
2LGV-1_epi54-57_L250
2LGV-2_epi54-57_L100
2LGV-2_epi54-57_L150
2LGV-2_epi54-57_L200
2LGV-2_epi54-57_L250
2LGV-0_epi87-96_L100
2LGV-0_epi87-96_L150
2LGV-0_epi87-96_L200
2LGV-0_epi87-96_L250
2LGV-1_epi87-96_L100
2LGV-1_epi87-96_L150
2LGV-1_epi87-96_L200
2LGV-1_epi87-96_L250
2LGV-2_epi87-96_L100
2LGV-2_epi87-96_L150
2LGV-2_epi87-96_L200
2LGV-2_epi87-96_L250

36 data versions.
aa torch.Size([1, 350])
pos_heavyatom torch.Size([1, 350, 4, 3])
mask_heavyatom torch.Size([1, 350, 4])
fragment_type torch.Size([1, 350])
mask torch.Size([1, 350])
generate_flag torch.Size([1, 350])
chain_nb torch.Size([1, 3

# Model

In [21]:
model_dir = '../checkpoints/'
model_list = [
    'JointDiff-binder_withEpi-withSite_randomMask_withSfold.pt',
    'JointDiff-binder_withEpi-withSite-withMono.pt',
    'JointDiff-x-binder_withEpi-withSite_randomMask_withSfold.pt',
    'JointDiff-x-binder_withEpi-withSite_withDist.pt',
]

In [22]:
model_dict = {}

for version in model_list:
    name = '.pt'.join(version.split('.pt')[:-1])
    model_path = os.path.join(model_dir, version)
    model_dict[name] = model_loading(model_path)

../checkpoints/JointDiff-binder_withEpi-withSite_randomMask_withSfold.pt
dict_keys(['config', 'model', 'optimizer', 'scheduler', 'iteration', 'avg_val_loss'])
../checkpoints/JointDiff-binder_withEpi-withSite-withMono.pt
dict_keys(['config', 'model', 'optimizer', 'scheduler', 'iteration', 'avg_val_loss'])
../checkpoints/JointDiff-x-binder_withEpi-withSite_randomMask_withSfold.pt
dict_keys(['config', 'model', 'optimizer', 'scheduler', 'iteration', 'avg_val_loss'])
../checkpoints/JointDiff-x-binder_withEpi-withSite_withDist.pt
dict_keys(['config', 'model', 'optimizer', 'scheduler', 'iteration', 'avg_val_loss'])


# Inference

In [126]:
device = 'cuda'
model = model.to(device)

### Test

In [48]:
_ = inference(
    model = model_dict['JointDiff-x-binder_withEpi-withSite_withDist'], 
    batch = data_dict['2LGV-0_epi43-46_L100'],
    attempts = 3,
    print_seq = True,
)

  0%|          | 0/3 [00:00<?, ?it/s]

*********** A **************
GGGGTNSGAGKKRFEVKKSNASAQSAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHCISRWLKTRQVCPLDNREWEFQKYGH
*********** B **************
ETREPVFPRRRITRRVTSPDTGEEAIAVAEKEAVTEERAFDPSPGPPAGGKDVFLLVVRFVDEPSRTEDDIELAVKIEDDTDEVVRVVGVVEASLGVGRL
*********** A **************
GGGGTNSGAGKKRFEVKKSNASAQSAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHCISRWLKTRQVCPLDNREWEFQKYGH
*********** B **************
AREGTKLEAAEGIATFGASVGKFLPVAEVAGDGERIKGILIVDGADAGVFFAGTSSDTDDPVAESATIRREADGVLPAKALLARDDAVGIVGTPPVEGTS
*********** A **************
GGGGTNSGAGKKRFEVKKSNASAQSAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHCISRWLKTRQVCPLDNREWEFQKYGH
*********** B **************
SGKTRFIRTAGAERGVASGGAIRRRELSGGAPRFARAKLIASSGEVPSGPAAGEDVGVVVLVLGDTREASFKVLTRVGKRKIAGLVAGRFGVAVEVSFDT


### inference

In [ ]:
device = 'cuda'
attempts = 10
result_dir = "../results/RBX1Binder/"

for m_name in model_dict:
    model = model_dict[m_name]
    print(f'*************** {m_name} ******************')

    for data_version in data_dict:
        print(f'### {data_version}')

        job_name = f'{m_name}_{data_version}'
        result_path = os.path.join(result_dir, 'dict', '%s.pkl' % job_name)
        pdb_dir = os.path.join(result_dir, 'pdbs', job_name)
        
        _ = inference(
            model = model, 
            batch = data_dict[data_version],
            device = device,
            attempts = attempts,
            result_path = result_path,
            save_pdb = True,
            pdb_dir = pdb_dir,
            save_name = job_name,
            print_seq = False,
        )

    print()

*************** JointDiff-binder_withEpi-withSite_randomMask_withSfold ******************
### 2LGV-0_epi43-46_L100


  0%|          | 0/10 [00:00<?, ?it/s]

### 2LGV-0_epi43-46_L150


  0%|          | 0/10 [00:00<?, ?it/s]

### 2LGV-0_epi43-46_L200


...

  0%|          | 0/10 [00:00<?, ?it/s]

### 2LGV-1_epi43-46_L250


  0%|          | 0/10 [00:00<?, ?it/s]

### 2LGV-2_epi43-46_L100


  0%|          | 0/10 [00:00<?, ?it/s]

# Filtering

In [20]:
def f1_score(pred, true, eps=1e-7):
    """
    Compute F1-score for binary classification using PyTorch.
    pred: (N,) tensor with predicted labels (0 or 1)
    true: (N,) tensor with ground truth labels (0 or 1)
    eps: small constant to avoid division by zero
    """
    pred = pred.int()
    true = true.int()

    # True Positives (TP), False Positives (FP), False Negatives (FN)
    tp = (pred * true).sum().float()
    fp = (pred * (1 - true)).sum().float()
    fn = ((1 - pred) * true).sum().float()

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)

    f1 = 2 * precision * recall / (precision + recall + eps)
    return f1

def seq_and_f1score(path, epitope_mask_true):

    model_id = 0
    
    structure = parser.get_structure("protein", path)[model_id]
    
    center_coor = {}
    seq_dict_coor = {}
    
    for chain in structure:
        chain_id = chain.get_id()
        center_coor[chain_id] = []
        seq_dict_coor[chain_id] = ''
    
        for res in chain:
            res_id = res.get_id()
            if res_id[0] != ' ':
                continue

            ### sequence
            res_idx = (str(res_id[1]) + res_id[2]).strip(' ')
            res_name = res.resname
            if res_name in RESIDUE_reverse_dict:
                res_letter = RESIDUE_reverse_dict[res_name]
            else:
                res_letter = 'X'
            seq_dict_coor[chain_id] += res_letter

            ### center_coor
            if 'CB' in res.child_dict:
                coor_center = res.child_dict['CB'].get_coord()
            else:
                coor_center = res.child_dict['CA'].get_coord()
            center_coor[chain_id].append(torch.from_numpy(coor_center).unsqueeze(0))
    
        center_coor[chain_id] = torch.cat(center_coor[chain_id], dim = 0)

    chain_dist = torch.cdist(center_coor['A'], center_coor['B'])
    inter_mask = chain_dist <= 8
    epitope_mask_pred = (torch.sum(inter_mask, dim = 1) > 0).int()

    score = f1_score(epitope_mask_pred, epitope_mask_true)
    
    return seq_dict_coor['B'], score

In [21]:
 seq_and_f1score(
     path = '../results/NipahBinder/pdbs/NipahBinder_JointDiff-binder_withEpi-withSite_randomMask_withSfold_motif106-125_attp10.pdb', 
     epitope_mask_true = epitope_mask
 )

('ALASMGVLHYPGKSLDDHHILLKVIVRLLASSNYGGVVTWIIIEITGQGYTYKILVVSGGLTIDFLLWLLVFKPDQDIKFTIKFQEFSPNLWDGTLTLLVVLAADEQGIILITYMQVEKLGLPKHEPLLTYYILFGT',
 tensor(0.7586))

In [26]:
pdb_dir = '../results/NipahBinder/pdbs/'
pdb_list = [p for p in os.listdir(pdb_dir) if p.endswith('.pdb')]
seq_score_list = []

for pdb in tqdm(pdb_list):
    name = pdb.split('/')[-1][:-4]
    path = os.path.join(pdb_dir, pdb)

    seq, score =  seq_and_f1score(
        path = path,
        epitope_mask_true = epitope_mask
    )
    seq_score_list.append((score, name, seq))

seq_score_list = sorted(seq_score_list, reverse = True)

  0%|          | 0/100 [00:00<?, ?it/s]

In [28]:
with open('../results/NipahBinder/samples_all.fa', 'w') as wf:
    for score, name, seq in seq_score_list:
        wf.write('>%s;f1=%.4f\n' % (name, score))
        wf.write('%s\n' % seq)